You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
# os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [7]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [8]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [9]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [10]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [11]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [12]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` enables detailed logs (newer crewAI uses bool, not `0/1/2`). 

In [14]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Jupyter**: Use `await crew.kickoff_async(...)` instead of `crew.kickoff()` — notebooks already run an asyncio event loop.

In [16]:
result = await crew.kickoff_async(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 04d97c27-93f7-45fe-abc4-5206ee049d4f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: 415b098e-8e95-43a5-b227-5879418ff59d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Content Strategy Plan: The State of Artificial Intelligence                                                  │
│                                                                                                                 │
│  **Date:** October 26, 2023 (Plan Creation Date)                                                                │
│  **Project:** Blog Article – Artificial Intelligence Overview                                                   │
│  **Prepared By:** Content Planner Team                                                                          │
│  **Status:** Ready for Content Writer                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Project Objective                                                                                        │
│  To produce an authoritative, engaging, and actionable blog article that demystifies the current state of       │
│  Artificial Intelligence. The goal is to help readers understand the rapid shifts in the AI landscape,          │
│  identify key market movers, and make informed decisions about how to leverage AI tools responsibly in their    │
│  personal or professional lives.                                                                                │
│                                                                                                                 │
│  ## 2. Target Audience Analysis                                                                                 │
│  We are targeting two primary personas. The content must balance accessibility for the layperson with enough    │
│  depth for the decision-maker.                                                                                  │
│                                                                                                                 │
│  ### Persona A: The Business Decision-Maker                                                                     │
│  *   **Demographics:** Ages 30–55, CTOs, Marketing Directors, Small Business Owners.                            │
│  *   **Interests:** Productivity gains, cost reduction, competitive advantage, automation tools.                │
│  *   **Pain Points:** Fear of being left behind, uncertainty about ROI, concerns over data privacy/security,    │
│  lack of clarity on which tools to trust.                                                                       │
│  *   **Desired Outcome:** A clear roadmap on where AI fits into their workflow without unnecessary jargon.      │
│                                                                                                                 │
│  ### Persona B: The Tech-Curious Professional                                                                   │
│  *   **Demographics:** Ages 22–40, Creators, Developers, Lifelong Learners.                                     │
│  *   **Interests:** New technologies, generative art/text, ethical implications, future of work.                │
│  *   **Pain Points:** Information overload, hype vs. re

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Beyond the Hype: Navigating the Current State of Artificial Intelligence Trends 2024                         │
│                                                                                                                 │
│  If you have used a smart assistant to draft an email today, you have already participated in one of the most   │
│  significant technological shifts of our lifetime. Yet, while headlines often scream about robots taking over   │
│  jobs, the reality is far more nuanced and immediately relevant to daily productivity. As we analyze the        │
│  **Current State of AI**, it becomes evident that technology has moved past the theoretical stage and is now    │
│  functioning as a utility comparable to electricity or the internet. Understanding **Artificial Intelligence    │
│  Trends 2024** is no longer optional for professionals who wish to remain competitive in an evolving digital    │
│  landscape.                                                                                                     │
│                                                                                                                 │
│  The rapid pace of change since the release of foundational Large Language Models (LLMs) has left many          │
│  organizations scrambling to catch up. We are witnessing a transition where AI is becoming embedded into the    │
│  fabric of workflows rather than existing as a standalone novelty. The urgency to understand this landscape is  │
│  driven less by fear of replacement and more by the necessity of leveraging these tools responsibly. This       │
│  distinction is critical for making informed adoption decisions that benefit both individuals and enterprises   │
│  alike.                                                                                                         │
│                                                                                                                 │
│  Ultimately, this article aims to demystify the noise surrounding modern technology. Whether evaluating         │
│  infrastructure or exploring new creative mediums, clarity is power. By separating fact from fiction,           │
│  stakeholders can navigate the opportunities presented by machine learning without falling prey to marketing    │
│  hype. The goal is to provide a roadmap that helps harness the capabilities of **Generative AI Use Cases**      │
│  effectively, ensuring that teams remain at the forefront of innovation rather than being swept away by it.     │
│                                                                                                                 │
│  ## How to Implement AI in Business: The Shift from Experimentation to Implementation                           │
│                                                                                                                 │
│  For the past two years, most organizations have treated AI as a playground for experimentation. However, a     │
│  definitive pivot toward real-world integration is underway, where efficiency metrics matter more than          │
│  novelty. According to Gartner reports, over 80% of enterprises were experimenting with GenAI by the end of     │
│  2023, but the focus is shifting from sandbox testing to production environments. This indicates a maturity in  │
│  the market where stakeholders demand measurable return

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 04d97c27-93f7-45fe-abc4-5206ee049d4f                                                                       │
│  Final Output: # Beyond the Hype: Navigating the Current State of Artificial Intelligence Trends 2024           │
│                                                                                                                 │
│  If you have used a smart assistant to draft an email today, you have already participated in one of the most   │
│  significant technological shifts of our lifetime. Yet, while headlines often scream about robots taking over   │
│  jobs, the reality is far more nuanced and immediately relevant to daily productivity. As we analyze the        │
│  **Current State of AI**, it becomes evident that technology has moved past the theoretical stage and is now    │
│  functioning as a utility comparable to electricity or the internet. Understanding **Artificial Intelligence    │
│  Trends 2024** is no longer optional for professionals who wish to remain competitive in an evolving digital    │
│  landscape.                                                                                                     │
│                                                                                                                 │
│  The rapid pace of change since the release of foundational Large Language Models (LLMs) has left many          │
│  organizations scrambling to catch up. We are witnessing a transition where AI is becoming embedded into the    │
│  fabric of workflows rather than existing as a standalone novelty. The urgency to understand this landscape is  │
│  driven less by fear of replacement and more by the necessity of leveraging these tools responsibly. This       │
│  distinction is critical for making informed adoption decisions that benefit both individuals and enterprises   │
│  alike.                                                                                                         │
│                                                                                                                 │
│  Ultimately, this article aims to demystify the noise surrounding modern technology. Whether evaluating         │
│  infrastructure or exploring new creative mediums, clarity is power. By separating fact from fiction,           │
│  stakeholders can navigate the opportunities presented by machine learning without falling prey to marketing    │
│  hype. The goal is to provide a roadmap that helps harness the capabilities of **Generative AI Use Cases**      │
│  effectively, ensuring that teams remain at the forefront of innovation rather than being swept away by it.     │
│                                                                                                                 │
│  ## How to Implement AI in Business: The Shift from Experimentation to Implementation                           │
│                                                                                                                 │
│  For the past two years, most organizations have treated AI as a playground for experimentation. However, a     │
│  definitive pivot toward real-world integration is underway, where efficiency metrics matter more than          │
│  novelty. According to Gartner reports, over 80% of enterprises were experimenting with GenAI by the end of     │
│  2023, but the focus is shifting from sandbox testing to production environments. This indicates a maturity in  │
│  the market where stakeholders demand measurable retur

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = await crew.kickoff_async(inputs={"topic": topic})

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).